# Eksperimen Static Embedding

Notebook ini menjalankan eksperimen **Word2Vec**, **FastText**, dan **GloVe** menggunakan pipeline yang tersedia di `src.models.static_embeddings`. Setiap eksperimen membentuk embedding harian, menggabungkannya dengan fitur leksikon, melatih model klasifikasi dan regresi, lalu menyimpan metrik dan artefak.

Jalankan sel secara berurutan dari root repository atau dari direktori lain; notebook akan mencari root repository secara otomatis. Pastikan dataset hasil preprocessing, split, dan feature extraction sudah tersedia. Training Word2Vec/FastText dapat memakan waktu. Pada pemakaian pertama, GloVe dapat mengunduh arsip Stanford sekitar **822 MB** jika cache dan file lokal tidak tersedia. Lokasi cache dan izin unduhan dapat diatur di sel konfigurasi.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

# Cari repository dari working directory kernel (termasuk parent directory).
PROJECT_ROOT = next(
    (
        candidate.resolve()
        for candidate in (Path.cwd(), *Path.cwd().resolve().parents)
        if (candidate / "src" / "models" / "static_embeddings" / "__init__.py").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Root repository tidak ditemukan. Jalankan notebook dari dalam repository."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.static_embeddings import run_embedding_experiment

print(f"Repository: {PROJECT_ROOT}")

Repository: /home/ajipratama53/Projects/nlp-trio-bertiga


## Konfigurasi

`MODELS_TO_RUN` secara default menjalankan ketiga model. Untuk menjalankan sebagian, hapus nama model yang tidak diperlukan dari daftar. `GLOVE_PATH` dapat diarahkan ke file GloVe lokal (teks 100d atau arsip `.zip`). Mengizinkan unduhan GloVe berarti notebook dapat mengunduh arsip Stanford berukuran sekitar 822 MB.

In [2]:
ARTICLES_PATH = PROJECT_ROOT / "data" / "processed" / "articles_aligned.csv"
ARTICLE_SPLITS_PATH = PROJECT_ROOT / "data" / "split" / "articles_split.csv"
MODEL_DATASET_PATH = PROJECT_ROOT / "data" / "feature_extracted" / "model_dataset.csv"
FEATURE_DIR = PROJECT_ROOT / "data" / "feature_extracted"
RESULTS_DIR = PROJECT_ROOT / "results"

MODELS_TO_RUN = ["word2vec", "fasttext", "glove"]

# Gunakan Path ke sumber GloVe lokal untuk menghindari unduhan arsip besar.
# Contoh: PROJECT_ROOT / "data" / "features" / "pretrained" / "glove.6B.zip"
GLOVE_PATH = None
GLOVE_CACHE_PATH = PROJECT_ROOT / "data" / "features" / "artifacts" / "glove_6b_100d.npz"
GLOVE_ARCHIVE_PATH = PROJECT_ROOT / "data" / "features" / "pretrained" / "glove.6B.zip"
ALLOW_GLOVE_DOWNLOAD = True  # GloVe dapat mengunduh arsip Stanford ~822 MB bila belum tersedia.

SUPPORTED_MODELS = {"word2vec", "fasttext", "glove"}
unknown_models = set(MODELS_TO_RUN) - SUPPORTED_MODELS
if unknown_models:
    raise ValueError(f"Model tidak dikenal: {sorted(unknown_models)}")
if not MODELS_TO_RUN:
    raise ValueError("Pilih minimal satu model di MODELS_TO_RUN.")

required_files = [ARTICLES_PATH, ARTICLE_SPLITS_PATH, MODEL_DATASET_PATH]
missing_files = [path for path in required_files if not path.is_file()]
if missing_files:
    missing_list = "\n".join(f"- {path}" for path in missing_files)
    raise FileNotFoundError(f"File input eksperimen tidak ditemukan:\n{missing_list}")

print("Model yang akan dijalankan:", ", ".join(MODELS_TO_RUN))
print("Input tersedia:")
for path in required_files:
    print(f"- {path}")
if "glove" in MODELS_TO_RUN:
    print(f"GloVe download diizinkan: {ALLOW_GLOVE_DOWNLOAD}")
    if GLOVE_PATH is None and not GLOVE_CACHE_PATH.exists() and not GLOVE_ARCHIVE_PATH.exists():
        print("PERINGATAN: GloVe akan mengunduh arsip Stanford sekitar 822 MB.")

Model yang akan dijalankan: word2vec, fasttext, glove
Input tersedia:
- /home/ajipratama53/Projects/nlp-trio-bertiga/data/processed/articles_aligned.csv
- /home/ajipratama53/Projects/nlp-trio-bertiga/data/split/articles_split.csv
- /home/ajipratama53/Projects/nlp-trio-bertiga/data/feature_extracted/model_dataset.csv
GloVe download diizinkan: True


## Menjalankan eksperimen

Setiap model menggunakan split train/validation/test kronologis yang sudah dibuat pipeline. Parameter embedding dan XGBoost memakai nilai default dari modul eksperimen. Hasil ditulis ke `results/`, fitur embedding harian ke `data/feature_extracted/`, dan cache/model ke direktori artefak yang ditentukan pipeline.

In [ ]:
common_args = [
    "--articles", str(ARTICLES_PATH),
    "--article-splits", str(ARTICLE_SPLITS_PATH),
    "--model-dataset", str(MODEL_DATASET_PATH),
    "--feature-dir", str(FEATURE_DIR),
    "--results-dir", str(RESULTS_DIR),
]

for model_name in MODELS_TO_RUN:
    print(f"\n{'=' * 20} {model_name.upper()} {'=' * 20}")
    args = list(common_args)
    if model_name == "glove":
        args.extend(["--glove-cache", str(GLOVE_CACHE_PATH)])
        args.extend(["--archive-path", str(GLOVE_ARCHIVE_PATH)])
        if GLOVE_PATH is not None:
            args.extend(["--glove-path", str(GLOVE_PATH)])
        if not ALLOW_GLOVE_DOWNLOAD:
            args.append("--no-download")

    status = run_embedding_experiment(model_name, args)
    if status != 0:
        raise RuntimeError(f"Eksperimen {model_name} gagal dengan status {status}.")

print("\nSemua eksperimen yang dipilih selesai.")

## Ringkasan hasil

Tabel perbandingan mencakup model static embedding dan baseline naive pada validation serta test, untuk klasifikasi dan regresi. File JSON menyimpan ringkasan metrik dan lokasi artefak masing-masing model.

In [ ]:
prefixes = {
    "word2vec": "07_word2vec",
    "fasttext": "08_fasttext",
    "glove": "09_glove",
}

comparison_tables = []
result_summaries = []
for model_name in MODELS_TO_RUN:
    prefix = prefixes[model_name]
    comparison_path = RESULTS_DIR / f"{prefix}_comparison.csv"
    results_path = RESULTS_DIR / f"{prefix}_results.json"
    if comparison_path.is_file():
        comparison = pd.read_csv(comparison_path)
        comparison.insert(0, "embedding", model_name)
        comparison_tables.append(comparison)
    if results_path.is_file():
        with results_path.open(encoding="utf-8") as result_file:
            result = json.load(result_file)
        result_summaries.append(
            {
                "embedding": model_name,
                "model": result.get("model"),
                "vocabulary_size": result.get("embedding", {}).get("vocabulary_size"),
                "best_iteration_clf": result.get("best_iteration", {}).get("clf"),
                "best_iteration_reg": result.get("best_iteration", {}).get("reg"),
                "results_file": str(results_path),
            }
        )

if comparison_tables:
    display(pd.concat(comparison_tables, ignore_index=True))
else:
    print("Belum ada file perbandingan; jalankan sel eksperimen terlebih dahulu.")

if result_summaries:
    display(pd.DataFrame(result_summaries))